The part i am working on is the speech rec to text pipeline , i don't perfect the recgontion yet

In [1]:
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration
)

: 

i used pytorch because the whisper model was made in it and it was the easiest model to use for now i guess

In [ ]:
dataset = load_dataset("MAdel121/arabic-egy-cleaned",streaming=True)

In [ ]:
print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

In [ ]:
audio = dataset["train"][0]["audio"]

In [ ]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small"
)

model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

In [ ]:
sample = next(iter(dataset["train"]))

audio = sample["audio"]

ground_truth = sample["text"]

print("Ground Truth:")
print(ground_truth)

In [ ]:
inputs = processor(
    audio["array"],
    sampling_rate=audio["sampling_rate"],
    return_tensors="pt"
)

input_features = inputs.input_features.to(device)

In [ ]:
predicted_ids = model.generate(input_features)

prediction = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

In [ ]:
print("Ground Truth:")
print(ground_truth)

print()

print("Whisper Prediction:")
print(prediction)

Evaluate based on wer and cer on multiple samples

In [ ]:
!pip install -q evaluate jiwer

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

In [ ]:
reference = sample["text"]
prediction = prediction.strip()

wer = wer_metric.compute(
    predictions=[prediction],
    references=[reference]
)

cer = cer_metric.compute(
    predictions=[prediction],
    references=[reference]
)

print("Ground Truth:")
print(reference)

print("\nPrediction:")
print(prediction)

print(f"\nWER : {wer:.3f}")
print(f"CER : {cer:.3f}")

In [ ]:
predictions = []
references = []

num_examples = 20
dataset_iter = iter(dataset["train"])

for i in range(num_examples):

    sample = next(dataset_iter)

    audio = sample["audio"]["array"]

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.no_grad():
        predicted_ids = model.generate(inputs.input_features.to(device))

    prediction = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    predictions.append(prediction)
    references.append(sample["text"])

In [ ]:
wer = wer_metric.compute(
    predictions=predictions,
    references=references
)

cer = cer_metric.compute(
    predictions=predictions,
    references=references
)

print("=" * 50)
print(f"Average WER : {wer:.3f}")
print(f"Average CER : {cer:.3f}")

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Ground Truth": references,
    "Prediction": predictions
})

results.head(10)

In [ ]:
sample_scores = []

for ref, pred in zip(references, predictions):

    score = wer_metric.compute(
        predictions=[pred],
        references=[ref]
    )

    sample_scores.append(score)

results["WER"] = sample_scores

results.sort_values(
    by="WER",
    ascending=False
).head(10)

The summary pipeline is next

In [ ]:
import re

from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

In [ ]:
# Preprocessing Functions
def remove_repeated_letters(text):
    return re.sub(r'(.)\1{2,}', r'\1', text)


def remove_extra_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()


def remove_diacritics(text):
    arabic_diacritics = re.compile("""
                             ّ|َ|ً|ُ|ٌ|ِ|ٍ|ْ|ـ
                         """, re.VERBOSE)

    return re.sub(arabic_diacritics, '', text)

In [ ]:
def preprocess(text):

    # Remove repeated letters
    text = remove_repeated_letters(text)

    # Remove Arabic diacritics
    text = remove_diacritics(text)

    # Remove extra spaces
    text = remove_extra_spaces(text)

    return text

In [ ]:
raw_text = """
 meetinggg بكرةةةة الساعة 10 يا جماااعة احنا هنعمل
"""

processed_text = preprocess(raw_text)

print(processed_text)

In [ ]:
# Text Summarization
def remove_fillers(text):

    fillers = [
        "يعني", "امم", "ااه", "آه", "بص", "بصي",
        "طيب", "طب", "ماشي", "تمام", "اوكي",
        "ممم", "اه", "همم"
    ]

    pattern = r'\b(?:' + '|'.join(fillers) + r')\b'

    text = re.sub(pattern, '', text)

    text = remove_extra_spaces(text)

    return text

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
def summarize(text):

    text = remove_fillers(text)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    summary_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        min_new_tokens=20,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

In [ ]:
raw_text = """
احنا اتفقنا ان الاجتماع هيكون الخميس الساعة عشرة.
احمد هيبعت التقرير.
سارة هتحجز القاعة.
"""

processed_text = preprocess(raw_text)

summary = summarize(processed_text)

print(summary)

In [ ]:
def extract_action_items(text):

    action_verbs = [
        "هيبعت", "هيجهز", "هتحجز", "هيعمل",
        "هيراجع", "هيكتب", "هيجهز", "هيتواصل",
       "دوره", "هيقوم", "هيبعتلنا", "هيخلص", "هيحدث", "هيعدل"
    ]

    sentences = re.split(r'[.!؟\n]', text)

    action_items = []

    for sentence in sentences:

        sentence = sentence.strip()

        if any(verb in sentence for verb in action_verbs):
            action_items.append(sentence)

    return action_items

In [ ]:
actions = extract_action_items(raw_text)

print(actions)

In [ ]:
# Final Func
def process_transcript(text):

    processed_text = preprocess(text)

    summary = summarize(processed_text)

    action_items = extract_action_items(processed_text)

    return {
        "processed_text": processed_text,
        "summary": summary,
        "action_items": action_items
    }

In [ ]:
test_text = """
احنا اتفقنا إن الاجتماع هيكون الخميس الساعة عشرة.
أحمد هيبعت التقرير.
سارة هتحجز القاعة.
محمد هيجهز البرزنتيشن.
"""

result = process_transcript(test_text)

print("\nOriginal Text:")
print(test_text)

print("\n\nProcessed Text:\n")
print(result["processed_text"])

print("\n\nSummary:\n")
print(result["summary"])

print("\n\nAction Items:\n")

for item in result["action_items"]:
    print("-", item)